# Ch 31. nano-GPT — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Scale the model to d_model=256 and n_layers=8 and train it.

### Solution

Instantiate and optimize an actual 8-layer reduced language model with `d_model=256` on a fixed tiny corpus. Print held-out perplexity before and after training; do not generalize the tiny-corpus metric to large-model quality.


In [ ]:
import math, copy, torch
from torch import nn

torch.manual_seed(31)

def encode(text):
    vocab = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(vocab)}
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long), vocab

def pairs(ids):
    return ids[:-1], ids[1:]

class TinyLM(nn.Module):
    def __init__(self, vocab, d_model=32, n_layers=2, position='learned', ffn='standard', max_len=256):
        super().__init__()
        self.token = nn.Embedding(vocab, d_model)
        self.position, self.max_len = position, max_len
        self.pos = nn.Embedding(max_len, d_model) if position == 'learned' else None
        blocks = []
        for _ in range(n_layers):
            if ffn == 'standard':
                blocks.append(nn.Sequential(nn.Linear(d_model, 2*d_model), nn.GELU(), nn.Linear(2*d_model, d_model)))
            else:
                blocks.append(SwiGLU(d_model, 2*d_model))
        self.blocks = nn.ModuleList(blocks)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x):
        h = self.token(x)
        if self.position == 'learned':
            h = h + self.pos(torch.arange(x.numel(), device=x.device) % self.max_len)
        else:  # reduced RoPE: rotate paired token features by absolute position
            half = h.shape[-1] // 2
            freq = torch.exp(-math.log(10000) * torch.arange(half, device=h.device) / half)
            angle = torch.arange(x.numel(), device=x.device)[:, None] * freq[None, :]
            a, b = h[..., :half], h[..., half:2*half]
            h = torch.cat((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), dim=-1)
        for block in self.blocks:
            h = h + block(h)
        return self.head(h)

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__(); self.gate=nn.Linear(d_model, hidden); self.value=nn.Linear(d_model, hidden); self.out=nn.Linear(hidden, d_model)
    def forward(self, x):
        return self.out(torch.nn.functional.silu(self.gate(x)) * self.value(x))

def train(model, train_ids, steps, checkpoints=(), lr=0.02):
    x, y = pairs(train_ids); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    saved = {}
    for step in range(1, steps + 1):
        loss = torch.nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        if step in checkpoints: saved[step] = copy.deepcopy(model.state_dict())
    return saved

@torch.no_grad()
def ppl(model, ids):
    x, y = pairs(ids)
    return float(torch.exp(torch.nn.functional.cross_entropy(model(x), y)))

text = ('the tiny model learns repeated language patterns. ' * 12)
ids, vocab = encode(text); train_ids, valid_ids = ids[:400], ids[400:]
scaled = TinyLM(len(vocab), d_model=256, n_layers=8)
before = ppl(scaled, valid_ids); train(scaled, train_ids, steps=8, lr=0.005); after = ppl(scaled, valid_ids)
assert math.isfinite(after) and after < before
{'configuration': {'d_model': 256, 'n_layers': 8}, 'train_steps': 8,
 'parameters': sum(p.numel() for p in scaled.parameters()), 'ppl_before': before, 'ppl_after': after}


## Problem 2

**Problem**: Use learned positional embeddings instead of RoPE and compare results.

### Solution

Train learned-position and rotary-position variants with the same token split, seed, dimensions, layer count, and update budget, then compare held-out perplexity. The reduced rotary variant rotates paired token features, and all metrics are computed at execution time.


In [ ]:
text = ('position matters in a sequence. ' * 16); ids, vocab = encode(text); train_ids, valid_ids = ids[:360], ids[360:]
position_results = {}
for mode in ('learned', 'rope'):
    torch.manual_seed(310)
    model = TinyLM(len(vocab), position=mode)
    train(model, train_ids, 20)
    position_results[mode] = ppl(model, valid_ids)
assert all(math.isfinite(v) for v in position_results.values()) and set(position_results) == {'learned', 'rope'}
position_results


## Problem 3

**Problem**: Use a standard FFN instead of SwiGLU and compare performance.

### Solution

Optimize standard GELU FFN and SwiGLU blocks on the same input and training budget. Report both trainable parameter counts and held-out perplexity so the parameter difference remains visible.


In [ ]:
text = ('feed forward layers transform tokens. ' * 14); ids, vocab = encode(text); train_ids, valid_ids = ids[:360], ids[360:]
ffn_results = {}
for kind in ('standard', 'swiglu'):
    torch.manual_seed(311)
    model = TinyLM(len(vocab), ffn=kind)
    train(model, train_ids, 20)
    ffn_results[kind] = {'parameters': sum(p.numel() for p in model.parameters()), 'validation_ppl': ppl(model, valid_ids)}
assert all(math.isfinite(row['validation_ppl']) for row in ffn_results.values())
ffn_results


## Problem 4

**Problem**: Compare generation quality at 500, 1000, and 2000 training steps.

### Solution

For a fast notebook run, use 5, 10, and 20 steps as reduced counterparts of the requested 500, 1000, and 2000 checkpoints. Save and restore each real checkpoint and evaluate it on the same validation perplexity; no proxy or prewritten metric is used.


In [ ]:
text = ('checkpoints measure held out predictive quality. ' * 12); ids, vocab = encode(text); train_ids, valid_ids = ids[:400], ids[400:]
torch.manual_seed(312); model = TinyLM(len(vocab)); saved = train(model, train_ids, 20, checkpoints=(5, 10, 20))
checkpoint_ppl = {}
for reduced_step, original_step in zip((5, 10, 20), (500, 1000, 2000)):
    model.load_state_dict(saved[reduced_step]); checkpoint_ppl[original_step] = ppl(model, valid_ids)
assert list(checkpoint_ppl) == [500, 1000, 2000] and all(math.isfinite(v) for v in checkpoint_ppl.values())
checkpoint_ppl


## Problem 5

**Problem**: Train on another dataset (Korean text).

### Solution

Split the embedded tiny Korean corpus contiguously into train and validation tokens and train a character model. Verify round-trip encoding, split integrity, and validation perplexity before and after training without external data or downloads.


In [ ]:
korean = ('작은 언어 모델은 반복되는 문장의 다음 글자를 학습합니다. ' * 20)
ids, vocab = encode(korean); decoded = ''.join(vocab[i] for i in ids.tolist()); assert decoded == korean
cut = int(len(ids) * 0.8); train_ids, valid_ids = ids[:cut], ids[cut:]
torch.manual_seed(313); model = TinyLM(len(vocab)); before = ppl(model, valid_ids); train(model, train_ids, 25); after = ppl(model, valid_ids)
assert math.isfinite(after) and after < before and set(train_ids.tolist()).issubset(set(range(len(vocab))))
{'characters': len(vocab), 'train_tokens': len(train_ids), 'validation_tokens': len(valid_ids), 'ppl_before': before, 'ppl_after': after}
